# Module 22 — Multi-Agent Debugging

Debug agent networks as distributed systems. Find the first causal failure, compare traces, attribute cost/latency and reconstruct security incidents.

In [ ]:
from dataclasses import dataclass, field


In [ ]:
@dataclass
class Event:
    id:str; run:str; causation:str|None; typ:str; component:str; ts:float; attrs:dict=field(default_factory=dict)

class Debugger:
    def __init__(self,events): self.events=sorted(events,key=lambda e:e.ts)
    def chain(self,event_id):
        by={e.id:e for e in self.events}; out=[]; cur=by.get(event_id)
        while cur:
            out.append(cur); cur=by.get(cur.causation) if cur.causation else None
        return list(reversed(out))
    def first_failure(self,ids):
        order={e.id:i for i,e in enumerate(self.events)}
        return min(ids,key=lambda x:order.get(x,999999)) if ids else None
    def costs(self):
        out={}
        for e in self.events: out[e.component]=out.get(e.component,0)+float(e.attrs.get('cost',0))
        return out


## 1. Build a causal trace
The final verifier failure is only the symptom. Trace backward through causation IDs.

In [ ]:
es=[Event('e1','r1',None,'run.started','orchestrator',1),Event('e2','r1','e1','agent.decided','supervisor',2,{'worker':'research'}),Event('e3','r1','e2','retrieval.completed','retriever',3,{'index':'old'}),Event('e4','r1','e3','verification.failed','verifier',4)]
d=Debugger(es)
[(e.id,e.typ) for e in d.chain('e4')]


## 2. First-failure principle
Suppose e3 has stale retrieval and e4 has a verification error. Identify e3 as the earlier causal failure.

In [ ]:
print('first failure:',d.first_failure(['e4','e3']))


## 3. Trace validation
Exercise: detect missing causation IDs, missing run IDs, invalid timestamps and malformed events.

In [ ]:
broken=Event('e5','r1','missing','tool.completed','tool',5)
known={e.id for e in es}
print('BROKEN TRACE' if broken.causation not in known else 'OK')


## 4. Cost attribution
Add token/tool costs to events and identify which component caused a spend spike.

In [ ]:
es += [Event('e5','r1','e4','tool.completed','tool',5,{'cost':.25}),Event('e6','r1','e5','model.completed','worker',6,{'cost':.80})]
print(d.costs())


## 5. Security incident
Inject a security denial after untrusted retrieval. Reconstruct how the malicious content reached the proposed action and verify policy blocked execution.

In [ ]:
es.append(Event('e7','r1','e3','security.denied','policy',7,{'reason':'untrusted content proposed restricted tool'}))
print([(e.id,e.typ) for e in d.chain('e7')])


## 6. Differential debugging
Create an expected trace and an actual trace. Compare worker selection, retrieval version, tool arguments and verification outcome.

In [ ]:
expected=['run.started','agent.decided','retrieval.completed','verification.completed']
actual=[e.typ for e in es[:4]]
print('expected:',expected); print('actual:',actual); print('diff:',[(i,a,b) for i,(a,b) in enumerate(zip(expected,actual)) if a!=b])


## Failure injection labs
1. Wrong worker routing.
2. Stale index.
3. Malformed worker output.
4. Mutated tool argument.
5. State-version corruption.
6. Duplicate side effect.
7. Retry storm.
8. Security-policy bypass attempt.
9. Missing trace event.
10. Cost explosion.
11. Latency regression.
12. Replay with safe fixtures.

# Exercises
Build a causal graph renderer; classify first failures; create trace diffs; calculate p95 component latency; attribute cost by agent; detect retry amplification; correlate security events; generate incident reports; create regression tests from incidents; integrate with Modules 20–21.

# Gold challenge
Build AegisAI Distributed Agent Debugger that ingests production-safe traces, reconstructs causal graphs, identifies the first invariant violation, attributes latency/cost/security impact, supports replay fixtures and emits a regression test plus incident report.